<a href="https://colab.research.google.com/github/Rakhayeva/multilingual-persuasion-detection/blob/main/03_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 03: Preprocessing

This notebook prepares the datasets for modeling:
- [Text cleaning](#) (removing URLs, extra whitespace)
- [Train / validation / test split for English (PTC)](#)
- KZ and RU datasets are reserved as test-only proxy sets

## Design Decision
Kazakh and Russian data are NOT included in training or validation.
They are used exclusively as test sets for cross-lingual transfer
experiments (H1). This is because:
1. Their task definition differs from EN (fake/real vs persuasion)
2. Their text length distribution differs substantially from EN
3. Including them in training would conflate two different tasks

In [1]:
# Connecting to Drive
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/NU/SEDS/multilingual-persuasion-detection'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Libraries
import os
import re
import pandas as pd
from sklearn.model_selection import train_test_split

In [3]:
# Load datasets
df_en = pd.read_csv(f'{BASE}/data/processed/ptc_en.csv')
df_kz = pd.read_csv(f'{BASE}/data/processed/baiangali_kz.csv')
df_ru = pd.read_csv(f'{BASE}/data/processed/baiangali_ru.csv')

print("Loaded:")
print(f"  EN: {len(df_en)} texts")
print(f"  KZ: {len(df_kz)} texts")
print(f"  RU: {len(df_ru)} texts")

Loaded:
  EN: 1147 texts
  KZ: 101 texts
  RU: 103 texts


## <a name='Cleaning'></a> Text Cleaning

In [4]:
def clean_text(text):
    """
    Basic text cleaning:
    - Remove URLs (not relevant to persuasion content)
    - Normalize whitespace (multiple spaces, newlines → single space)
    - Strip leading/trailing whitespace
    We intentionally do NOT lowercase or remove punctuation here,
    as transformer models benefit from original casing and punctuation.
    """
    if not isinstance(text, str):
        return ''
    text = re.sub(r'http\S+|www\S+', '', text)  # remove URLs
    text = re.sub(r'\s+', ' ', text).strip()     # normalize whitespace
    return text

# Apply cleaning to all datasets
for df in [df_en, df_kz, df_ru]:
    df['text_clean'] = df['text'].apply(clean_text)

# Remove texts that are too short after cleaning
before = len(df_en)
df_en = df_en[df_en['text_clean'].str.len() >= 50].copy()
print(f"EN: removed {before - len(df_en)} texts shorter than 50 chars")
print(f"EN remaining: {len(df_en)}")

# Check for any empty texts in KZ and RU
print(f"\nKZ empty texts: {(df_kz['text_clean'].str.len() < 20).sum()}")
print(f"RU empty texts: {(df_ru['text_clean'].str.len() < 20).sum()}")

EN: removed 81 texts shorter than 50 chars
EN remaining: 1066

KZ empty texts: 0
RU empty texts: 0


## 2. Train / Validation / Test Split

English (PTC) is split into:
- 70% train - for model training
- 15% validation - for hyperparameter tuning
- 15% test - for final evaluation

Stratification ensures class balance is preserved in each split.
Kazakh and Russian are kept as separate test-only sets.

In [6]:
# ── Train / Val / Test split for English ─────────────────────────────────────
# Stratify by has_persuasion to preserve class ratio across splits

df_train, df_temp = train_test_split(
    df_en,
    test_size=0.30,
    random_state=42,
    stratify=df_en['has_persuasion']
)

df_val, df_test_en = train_test_split(
    df_temp,
    test_size=0.50,
    random_state=42,
    stratify=df_temp['has_persuasion']
)

# KZ and RU are test-only — no splitting needed
df_test_kz = df_kz.copy()
df_test_ru = df_ru.copy()

# ── Verify split sizes and class balance ──────────────────────────────────────
print("Split sizes:")
print(f"  Train:    {len(df_train)} texts")
print(f"  Val:      {len(df_val)} texts")
print(f"  Test EN:  {len(df_test_en)} texts")
print(f"  Test KZ:  {len(df_test_kz)} texts (proxy)")
print(f"  Test RU:  {len(df_test_ru)} texts (proxy)")

print("\nClass balance in each split:")
for name, df in [('Train',   df_train),
                  ('Val',     df_val),
                  ('Test EN', df_test_en)]:
    pos = df['has_persuasion'].mean() * 100
    print(f"  {name:<10} manipulative: {pos:.1f}%")

Split sizes:
  Train:    746 texts
  Val:      160 texts
  Test EN:  160 texts
  Test KZ:  101 texts (proxy)
  Test RU:  103 texts (proxy)

Class balance in each split:
  Train      manipulative: 58.6%
  Val        manipulative: 58.8%
  Test EN    manipulative: 58.8%


Excellent, the balance remained the same in all splits (~58.8%), stratification worked correctly.

## Save Processed Datasets

In [7]:
# ── Save all splits to processed folder ──────────────────────────────────────
# We save text_clean (cleaned version) alongside original text
# and all metadata columns for traceability

df_train.to_csv(f'{BASE}/data/processed/train.csv', index=False)
df_val.to_csv(f'{BASE}/data/processed/val.csv', index=False)
df_test_en.to_csv(f'{BASE}/data/processed/test_en.csv', index=False)
df_test_kz.to_csv(f'{BASE}/data/processed/test_kz.csv', index=False)
df_test_ru.to_csv(f'{BASE}/data/processed/test_ru.csv', index=False)

print("Saved:")
print(f"  train.csv       → {len(df_train)} texts")
print(f"  val.csv         → {len(df_val)} texts")
print(f"  test_en.csv     → {len(df_test_en)} texts")
print(f"  test_kz.csv     → {len(df_test_kz)} texts (proxy)")
print(f"  test_ru.csv     → {len(df_test_ru)} texts (proxy)")

# Verify columns saved correctly
print(f"\nColumns in train.csv: {list(df_train.columns)}")

Saved:
  train.csv       → 746 texts
  val.csv         → 160 texts
  test_en.csv     → 160 texts
  test_kz.csv     → 101 texts (proxy)
  test_ru.csv     → 103 texts (proxy)

Columns in train.csv: ['article_id', 'text', 'has_persuasion', 'language', 'source', 'text_clean']


## Summary: Preprocessing Complete

| File | Split | Language | Task | Texts |
|------|-------|----------|------|-------|
| `train.csv` | Train | EN | Persuasion | 746 |
| `val.csv` | Validation | EN | Persuasion | 160 |
| `test_en.csv` | Test | EN | Persuasion | 160 |
| `test_kz.csv` | Test | KZ | Fake/Real (proxy) | 101 |
| `test_ru.csv` | Test | RU | Fake/Real (proxy) | 103 |

Class balance (~58.8% manipulative) is consistent across
train / val / test EN splits — stratification successful.

**Next:** [`04_baseline_models.ipynb`](https://colab.research.google.com/drive/1tqHlouKHNJYKNrciYk3qfbQ_idTQIHfD?usp=sharing) - TF-IDF + Logistic Regression / SVM